In [0]:
data = [
    (101,"Arjun Reddy","Hyderabad","Cardiology",5000),
    (102,"Sneha Kapoor","Delhi","Orthopedics",3000),
    (103,"Rahul Sharma","Mumbai","Dermatology",1500),
    (104,"Priya Nair","Bangalore","Cardiology",5000),
    (105,"Vikram Singh","Chennai","Neurology",7000)
]

columns = ["visit_id","patient_name","city","department","consultation_fee"]

df = spark.createDataFrame(data, columns)
display(df)

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
102,Sneha Kapoor,Delhi,Orthopedics,3000
103,Rahul Sharma,Mumbai,Dermatology,1500
104,Priya Nair,Bangalore,Cardiology,5000
105,Vikram Singh,Chennai,Neurology,7000


In [0]:
df.write \
  .mode("overwrite") \
  .parquet("/tmp/patient_parquet")
  

In [0]:
parquet_df = spark.read.parquet("/tmp/patient_parquet")
display(parquet_df)

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
104,Priya Nair,Bangalore,Cardiology,5000
103,Rahul Sharma,Mumbai,Dermatology,1500
105,Vikram Singh,Chennai,Neurology,7000
102,Sneha Kapoor,Delhi,Orthopedics,3000


In [0]:
parquet_df.printSchema()

root
 |-- visit_id: long (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- department: string (nullable = true)
 |-- consultation_fee: long (nullable = true)



In [0]:
spark.read.parquet("/tmp/patient_parquet") \
  .select("patient_name","city") \
  .show()

+------------+---------+
|patient_name|     city|
+------------+---------+
| Arjun Reddy|Hyderabad|
|  Priya Nair|Bangalore|
|Rahul Sharma|   Mumbai|
|Vikram Singh|  Chennai|
|Sneha Kapoor|    Delhi|
+------------+---------+



In [0]:
spark.read.parquet("/tmp/patient_parquet") \
  .filter("consultation_fee > 3000") \
  .show()

+--------+------------+---------+----------+----------------+
|visit_id|patient_name|     city|department|consultation_fee|
+--------+------------+---------+----------+----------------+
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|
|     104|  Priya Nair|Bangalore|Cardiology|            5000|
|     105|Vikram Singh|  Chennai| Neurology|            7000|
+--------+------------+---------+----------+----------------+



In [0]:
df.write \
  .mode("overwrite") \
  .partitionBy("city") \
  .parquet("/tmp/patient_parquet_partitioned")

In [0]:
spark.read.parquet("/tmp/patient_parquet_partitioned").show()

+--------+------------+-----------+----------------+---------+
|visit_id|patient_name| department|consultation_fee|     city|
+--------+------------+-----------+----------------+---------+
|     103|Rahul Sharma|Dermatology|            1500|   Mumbai|
|     102|Sneha Kapoor|Orthopedics|            3000|    Delhi|
|     101| Arjun Reddy| Cardiology|            5000|Hyderabad|
|     105|Vikram Singh|  Neurology|            7000|  Chennai|
|     104|  Priya Nair| Cardiology|            5000|Bangalore|
+--------+------------+-----------+----------------+---------+



In [0]:
spark.read.parquet("/tmp/patient_parquet_partitioned") \
  .filter("city = 'Hyderabad'") \
  .show()

+--------+------------+----------+----------------+---------+
|visit_id|patient_name|department|consultation_fee|     city|
+--------+------------+----------+----------------+---------+
|     101| Arjun Reddy|Cardiology|            5000|Hyderabad|
+--------+------------+----------+----------------+---------+



In [0]:
new_data = [(106,"Ananya Das","Kolkata","Orthopedics",3000)]

new_df = spark.createDataFrame(new_data, columns)

new_df.write \
  .mode("append") \
  .parquet("/tmp/patient_parquet")

In [0]:
df.write \
  .mode("overwrite") \
  .parquet("/tmp/patient_parquet")

In [0]:
display(dbutils.fs.ls("dbfs:/tmp/patient_parquet"))


path,name,size,modificationTime
dbfs:/tmp/patient_parquet/_SUCCESS,_SUCCESS,0,1777888049000
dbfs:/tmp/patient_parquet/_committed_3078470604483993900,_committed_3078470604483993900,524,1777887897000
dbfs:/tmp/patient_parquet/_committed_3600471320387984765,_committed_3600471320387984765,1133,1777888049000
dbfs:/tmp/patient_parquet/_committed_548010693257932682,_committed_548010693257932682,123,1777888035000
dbfs:/tmp/patient_parquet/_started_3078470604483993900,_started_3078470604483993900,0,1777887896000
dbfs:/tmp/patient_parquet/_started_3600471320387984765,_started_3600471320387984765,0,1777888049000
dbfs:/tmp/patient_parquet/_started_548010693257932682,_started_548010693257932682,0,1777888035000
dbfs:/tmp/patient_parquet/part-00000-tid-3600471320387984765-7e9bd2e4-2669-4d83-b230-a006a3b0143b-165-1.c000.snappy.parquet,part-00000-tid-3600471320387984765-7e9bd2e4-2669-4d83-b230-a006a3b0143b-165-1.c000.snappy.parquet,1618,1777888049000
dbfs:/tmp/patient_parquet/part-00001-tid-3600471320387984765-7e9bd2e4-2669-4d83-b230-a006a3b0143b-161-1.c000.snappy.parquet,part-00001-tid-3600471320387984765-7e9bd2e4-2669-4d83-b230-a006a3b0143b-161-1.c000.snappy.parquet,1605,1777888049000
dbfs:/tmp/patient_parquet/part-00002-tid-3600471320387984765-7e9bd2e4-2669-4d83-b230-a006a3b0143b-162-1.c000.snappy.parquet,part-00002-tid-3600471320387984765-7e9bd2e4-2669-4d83-b230-a006a3b0143b-162-1.c000.snappy.parquet,1611,1777888049000


In [0]:
%sql
CREATE OR REPLACE TABLE patient_parquet_table
USING PARQUET
LOCATION '/tmp/patient_parquet';

In [0]:
%sql
SELECT * FROM patient_parquet_table;

In [0]:
%sql
CONVERT TO DELTA parquet.`/tmp/patient_parquet`;

In [0]:
%sql
UPDATE delta.`/tmp/patient_parquet`
SET consultation_fee = 6000
WHERE visit_id = 101;